In [ ]:
# SETUP & IMPORTS

import sys
import time
import pathlib
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
if ROOT.name != "risk_assessment":
    raise RuntimeError(f"Expected ROOT to be 'risk_assessment', but got '{ROOT.name}'")

sys.path.insert(0, str(ROOT / "src"))

from features import (
    load_feature_contract,
    apply_filter_contract,
    fit_outlier_bounds,
    apply_clamping,
    apply_log1p_transform,
    assert_feature_integrity
)

print("Notebook 3.0 - Feature Engineering")
print(f"Project root : {ROOT}")
print(f"Polars       : {pl.__version__}")


Notebook 3.0 - Feature Engineering
Project root : /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment
Polars       : 1.41.2


In [ ]:
# INGESTION & FILTER CONTRACT

TRAIN_CLEAN = ROOT / "data" / "processed" / "train_clean.parquet"
TEST_CLEAN = ROOT / "data" / "processed" / "test_clean.parquet"
SHORTLIST = ROOT / "reports" / "model_feature_shortlist.csv"
TRAIN_OUT = ROOT / "data" / "processed" / "train_features.parquet"
TEST_OUT = ROOT / "data" / "processed" / "test_features.parquet"

train_raw = pl.read_parquet(TRAIN_CLEAN)
test_raw = pl.read_parquet(TEST_CLEAN)

print("STEP 1: INGESTION & FILTER CONTRACT")
print("-" * 50)
print(f"Original train shape : {train_raw.shape}")
print(f"Original test shape  : {test_raw.shape}")

keep_features = load_feature_contract(str(SHORTLIST))
train_filt = apply_filter_contract(train_raw, keep_features, split_tag="train")
test_filt = apply_filter_contract(test_raw, keep_features, split_tag="test")

expected_col_count = len(keep_features) + 1
assert train_filt.shape[1] == expected_col_count, f"train_filt has {train_filt.shape[1]} columns, expected {expected_col_count}"
assert test_filt.shape[1] == expected_col_count, f"test_filt has {test_filt.shape[1]} columns, expected {expected_col_count}"

print(f"Asserted both filtered DataFrames have exactly {expected_col_count} columns.")

TRAIN_ROWS = train_filt.shape[0]
TEST_ROWS = test_filt.shape[0]


STEP 1: INGESTION & FILTER CONTRACT
--------------------------------------------------
Original train shape : (402754, 78)
Original test shape  : (40207, 78)
[CONTRACT] Loaded 52 KEEP features from /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/model_feature_shortlist.csv
[FILTER] train : 78 cols -> 53 cols
[FILTER] test : 78 cols -> 53 cols
Asserted both filtered DataFrames have exactly 53 columns.


In [ ]:
# OUTLIER BOUNDS FITTING (TRAIN ONLY)

print("STEP 2: OUTLIER BOUNDS FITTING (TRAIN ONLY)")
print("-" * 50)

valid_dtypes = [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64]
numeric_cols = [
    col for col in train_filt.columns
    if train_filt.schema[col] in valid_dtypes and col != "target"
]

print(f"  Numeric columns selected for bounding: {len(numeric_cols)}")

outlier_bounds = fit_outlier_bounds(train_filt, numeric_cols)

print("Bounds dict sample (first 3 entries):")
for key, val in list(outlier_bounds.items())[:3]:
    print(f"    {key}: {val}")

print("Bounds fitted on train set. Test set untouched.")


STEP 2: OUTLIER BOUNDS FITTING (TRAIN ONLY)
--------------------------------------------------
  Numeric columns selected for bounding: 52
[OUTLIER FIT] Computed P1/P99 bounds for 52 numeric columns (train only)
Bounds dict sample (first 3 entries):
    risk_factor: {'lower': 1e-06, 'upper': 0.970361196761651}
    max_risk_factor: {'lower': 0.0, 'upper': 1.2046695010085142}
    risk_factor_above_threshold_daily_count: {'lower': 0.0, 'upper': 89897.37999999849}
Bounds fitted on train set. Test set untouched.


In [ ]:
# TRANSFORMATION LAYER

print("STEP 3: TRANSFORMATION LAYER")
print("-" * 50)

train_clamped = apply_clamping(train_filt, outlier_bounds, split_tag="train")
test_clamped = apply_clamping(test_filt, outlier_bounds, split_tag="test")

train_final = apply_log1p_transform(train_clamped, split_tag="train")
test_final = apply_log1p_transform(test_clamped, split_tag="test")

train_nulls = train_final.null_count().to_series().sum()
test_nulls = test_final.null_count().to_series().sum()

print("\n  split | rows     | cols | nulls")
print("  ------+----------+------+------")
print(f"  train | {train_final.shape[0]:<8} | {train_final.shape[1]:<4} | {train_nulls}")
print(f"  test  | {test_final.shape[0]:<8} | {test_final.shape[1]:<4} | {test_nulls}")

assert train_final.shape[1] == expected_col_count, "train col count mismatch"
assert test_final.shape[1] == expected_col_count, "test col count mismatch"


STEP 3: TRANSFORMATION LAYER
--------------------------------------------------
[CLAMP] train : applied clamping to 52 columns
[CLAMP] test : applied clamping to 52 columns
[LOG1P] train : applied log1p to 31 columns -> ['risk_factor_above_threshold_daily_count', 'max_eth_ever', 'min_eth_ever', 'withdraw_amount_sum_eth', 'repay_amount_avg_eth', 'repay_amount_sum_eth', 'liquidation_amount_sum_eth', 'total_balance_eth', 'liquidation_count', 'deposit_amount_sum_eth', 'risky_sum_outgoing_amount_eth', 'borrow_count', 'deposit_count', 'borrow_repay_diff_eth', 'total_available_borrows_avg_eth', 'repay_count', 'total_collateral_avg_eth', 'unique_borrow_protocol_count', 'borrow_amount_sum_eth', 'incoming_tx_count', 'outgoing_tx_sum_eth', 'risky_tx_count', 'borrow_amount_avg_eth', 'incoming_tx_sum_eth', 'total_available_borrows_eth', 'outgoing_tx_count', 'total_collateral_eth', 'total_gas_paid_eth', 'net_incoming_tx_count', 'avg_gas_paid_per_tx_eth', 'unique_lending_protocol_count']
[LOG1P] test

In [ ]:
# PERSISTENCE LAYER

print("STEP 4: PERSISTENCE LAYER")
print("-" * 50)

TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)

train_final.write_parquet(TRAIN_OUT, compression="snappy")
test_final.write_parquet(TEST_OUT, compression="snappy")

if TRAIN_OUT.exists():
    print(f"  [WRITE] train_features.parquet -> {TRAIN_OUT.stat().st_size // 1024} KB")
if TEST_OUT.exists():
    print(f"  [WRITE] test_features.parquet  -> {TEST_OUT.stat().st_size // 1024} KB")

print("  [OK] Both feature files written successfully.")

STEP 4: PERSISTENCE LAYER
--------------------------------------------------
  [WRITE] train_features.parquet -> 76244 KB
  [WRITE] test_features.parquet  -> 7484 KB
  [OK] Both feature files written successfully.


In [ ]:
# MLOPS ASSERTION GATE (RELOAD + VERIFY)

print("STEP 5: MLOPS ASSERTION GATE (RELOAD + VERIFY)")
print("-" * 50)
print("  Reloading files from disk for cold verification...")

train_check = pl.read_parquet(TRAIN_OUT)
test_check = pl.read_parquet(TEST_OUT)

assert_feature_integrity(
    train_check,
    expected_rows=TRAIN_ROWS,
    expected_cols=train_check.shape[1],
    split_tag="train",
)

assert_feature_integrity(
    test_check,
    expected_rows=TEST_ROWS,
    expected_cols=test_check.shape[1],
    split_tag="test",
)

print("NOTEBOOK 3.0 COMPLETE")
print("Pipeline outputs locked and verified.")
print(f"train_features.parquet : {train_check.shape[0]} rows x {train_check.shape[1]} cols")
print(f"test_features.parquet  : {test_check.shape[0]} rows x {test_check.shape[1]} cols")


STEP 5: MLOPS ASSERTION GATE (RELOAD + VERIFY)
--------------------------------------------------
  Reloading files from disk for cold verification...
[ASSERT] train rows   : PASS (402754 rows)
[ASSERT] train cols   : PASS (53 cols)
[ASSERT] train nulls  : PASS (0 nulls)
[ASSERT] train target : PASS (column exists)
[INTEGRITY] train : ALL CHECKS PASSED
[ASSERT] test rows   : PASS (40207 rows)
[ASSERT] test cols   : PASS (53 cols)
[ASSERT] test nulls  : PASS (0 nulls)
[ASSERT] test target : PASS (column exists)
[INTEGRITY] test : ALL CHECKS PASSED
NOTEBOOK 3.0 COMPLETE
Pipeline outputs locked and verified.
train_features.parquet : 402754 rows x 53 cols
test_features.parquet  : 40207 rows x 53 cols
